# Part C: Stronger OR Optimization for Warehouse Assignment

This notebook rebuilds Part C so the optimization model is structurally stronger than a simple weighted assignment score.

The upgraded Part C keeps the project course-appropriate:
- it reuses Part A processed data,
- it reuses the Analysis 3 candidate-level time proxy,
- it stays notebook-friendly and solvable with PuLP,
- it does not claim true inventory quantity or true capacity,
- and it reports honestly if stronger optimization still yields only modest gains.

The two main upgrades are:
1. a **lexicographic two-stage service policy**, and
2. a **workload-aware epsilon-constraint policy** with an explicit workload proxy.


## 2. Why the old model was too simple

The earlier Part C was too close to a basic weighted score minimization:

- one assignment decision per order line,
- one simple weighted objective,
- and no real policy structure beyond a time-plus-penalty ranking.

That is useful as a first pass, but it underuses OR ideas.

This notebook upgrades Part C in three ways:
- it uses a genuine observed-policy baseline instead of an optimizer-friendly fallback,
- it uses a **two-stage lexicographic formulation** rather than a single weighted sum,
- and it introduces a **workload proxy policy** that explicitly trades off service against DC workload pressure.


## 3. Observed-policy baseline definition

The baseline is the observed historical fulfillment warehouse `dc_ori`, not a constructed candidate-set fallback.

For each sampled order line, the observed policy is classified as:

- `historical_feasible`: `dc_ori` is in the candidate set and `inventory_available = 1`
- `historical_present_but_inventory_blocked`: `dc_ori` is in the candidate set but not inventory-feasible
- `historical_missing_from_candidate_set`: `dc_ori` is not in the current candidate set

The main baseline-vs-optimization comparison is restricted to the `historical_feasible` subset. This keeps the comparison honest and avoids silently replacing the baseline with a warehouse that is already favorable to the optimizer.


## 4. Optimization Formulation 1: Lexicographic Service Policy

**Structural upgrade 1: lexicographic optimization**

This formulation is intentionally different from a single weighted sum.

Stage 1:
- minimize total predicted fulfillment time

Stage 2:
- among solutions with total predicted time within a small tolerance of Stage 1,
- minimize the number of remote assignments

This represents a genuine policy statement:
- service remains the primary objective,
- but when multiple near-service-equivalent assignments exist,
- the model prefers less remote fulfillment.

The tolerance is reported explicitly so the trade-off remains transparent.


## 5. Optimization Formulation 2: Workload-Aware Proxy Policy

**Structural upgrade 2: epsilon-constraint plus workload proxy**

True warehouse capacity is not observed, so this notebook does **not** claim a real capacity model. Instead, it uses a workload proxy based on observed historical assignment counts by DC in the comparable baseline subset.

For each DC:

- observed baseline workload proxy = number of historically assigned comparable order lines
- proxy cap = `max(load_floor, (1 + gamma) * baseline_load)`
- overload variable = assignment load above that proxy cap

The workload-aware policy is solved in two stages:

Stage A:
- minimize total overload above proxy caps
- subject to total predicted time being no worse than the service optimum plus a small epsilon

Stage B:
- among overload-optimal solutions,
- minimize total predicted fulfillment time

This creates a mathematically different policy structure:
- the service-optimal policy asks “how fast can we be?”
- the workload-aware policy asks “how much workload pressure can we relieve while staying near the service frontier?”


In [1]:
%matplotlib inline

from pathlib import Path
import inspect

import matplotlib
import numpy as np
import pandas as pd
import pulp
from IPython.display import display

matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

ORDER_LINE_PATH = PROCESSED_DIR / "order_line_with_inventory.csv"
CANDIDATE_PATH = PROCESSED_DIR / "assignment_candidates.csv"

SAMPLE_DATES_N = 14
TOP_REGIONS_N = 5
MAX_SAMPLE_LINES = 3000
TIME_LIMIT_SECONDS = 60
EPSILON_TIE_BREAK = 1e-6

LEXI_TIME_TOLERANCE_TOTAL_HOURS = 50.0
WORKLOAD_TIME_TOLERANCE_TOTAL_HOURS = 20.0
WORKLOAD_GAMMA = 0.10
WORKLOAD_FLOOR_LOAD = 10.0


def normalize_code(series: pd.Series) -> pd.Series:
    numeric = pd.to_numeric(series, errors="coerce")
    return numeric.round().astype("Int64").astype("string")


def stable_holdout_mask(series: pd.Series, train_share: float = 0.8) -> pd.Series:
    hashed = pd.util.hash_pandas_object(series.astype("string"), index=False).astype("uint64")
    return (hashed % 10) < int(train_share * 10)


def make_onehot_encoder():
    params = inspect.signature(OneHotEncoder).parameters
    if "sparse_output" in params:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_modeling_sample(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(
        {
            "order_line_id": df["order_line_id"].astype("string"),
            "hours_to_delivery": pd.to_numeric(df.get("hours_to_delivery"), errors="coerce"),
            "remote_fulfillment_flag": df["dc_ori"].astype("string").ne(df["dc_des"].astype("string")).astype(float),
            "promise_days": pd.to_numeric(df.get("promise_days", df.get("promise")), errors="coerce"),
            "quantity": pd.to_numeric(df.get("quantity"), errors="coerce"),
            "discount_rate": pd.to_numeric(df.get("discount_rate"), errors="coerce"),
            "package_count": pd.to_numeric(df.get("package_count"), errors="coerce"),
            "inventory_at_fulfillment_dc": pd.to_numeric(df.get("inventory_at_dc_ori"), errors="coerce"),
            "num_available_dcs_in_region": pd.to_numeric(df.get("num_available_dcs_in_region"), errors="coerce"),
            "plus": pd.to_numeric(df.get("plus"), errors="coerce"),
            "user_level": pd.to_numeric(df.get("user_level"), errors="coerce"),
            "purchase_power": pd.to_numeric(df.get("purchase_power"), errors="coerce"),
            "city_level": pd.to_numeric(df.get("city_level"), errors="coerce"),
            "destination_region_id": df["destination_region_id"].astype("string"),
            "delivered_by_jd_flag": pd.to_numeric(df.get("delivered_by_jd_flag"), errors="coerce"),
        }
    )

    sku_type_raw = df.get("type")
    sku_type_num = pd.to_numeric(sku_type_raw, errors="coerce")
    sku_type_text = sku_type_raw.astype("string").fillna("Unknown").str.upper()
    out["sku_type"] = "Unknown"
    out.loc[sku_type_num.eq(1).fillna(False), "sku_type"] = "1P"
    out.loc[sku_type_num.eq(2).fillna(False), "sku_type"] = "3P"
    valid_text_mask = sku_type_text.isin(["1P", "3P"]).fillna(False)
    out.loc[valid_text_mask, "sku_type"] = sku_type_text.loc[valid_text_mask]

    out = out.loc[out["delivered_by_jd_flag"].fillna(1).eq(1)].copy()
    out = out.loc[out["hours_to_delivery"].notna()].copy()
    out = out.loc[out["hours_to_delivery"] > 0].copy()
    out = out.loc[out["hours_to_delivery"] <= 240].copy()

    out["promise_missing_flag"] = out["promise_days"].isna().astype(int)
    out["package_missing_flag"] = out["package_count"].isna().astype(int)
    out["log_hours_to_delivery"] = np.log1p(out["hours_to_delivery"])
    return out.reset_index(drop=True)


def add_business_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["sku_type_3p_flag"] = (out["sku_type"] == "3P").astype(float)
    out["remote_x_inventory_at_fulfillment_dc"] = out["remote_fulfillment_flag"] * out["inventory_at_fulfillment_dc"]
    out["remote_x_package_count"] = out["remote_fulfillment_flag"] * out["package_count"]
    out["remote_x_promise_days"] = out["remote_fulfillment_flag"] * out["promise_days"]
    out["remote_x_sku_type_3p"] = out["remote_fulfillment_flag"] * out["sku_type_3p_flag"]
    return out


def prepare_model_matrix(df: pd.DataFrame, numeric_features: list[str], categorical_features: list[str]) -> pd.DataFrame:
    out = df.copy().replace({pd.NA: np.nan})
    for col in numeric_features:
        out[col] = pd.to_numeric(out[col], errors="coerce")
    for col in categorical_features:
        out[col] = out[col].astype("object").where(pd.notna(out[col]), np.nan)
    return out


def fit_analysis3_proxy(order_line_raw: pd.DataFrame):
    model_df = add_business_features(build_modeling_sample(order_line_raw))
    train_mask = stable_holdout_mask(model_df["order_line_id"])

    numeric_features = [
        "remote_fulfillment_flag",
        "promise_days",
        "promise_missing_flag",
        "quantity",
        "discount_rate",
        "package_count",
        "package_missing_flag",
        "inventory_at_fulfillment_dc",
        "num_available_dcs_in_region",
        "plus",
        "user_level",
        "purchase_power",
        "city_level",
        "remote_x_inventory_at_fulfillment_dc",
        "remote_x_package_count",
        "remote_x_promise_days",
        "remote_x_sku_type_3p",
    ]
    categorical_features = ["sku_type", "destination_region_id"]
    feature_columns = numeric_features + categorical_features

    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_features,
            ),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", make_onehot_encoder()),
                    ]
                ),
                categorical_features,
            ),
        ]
    )

    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("ridge", RidgeCV(alphas=np.logspace(-2, 2, 10), cv=5)),
        ]
    )

    X_train = prepare_model_matrix(model_df.loc[train_mask, feature_columns], numeric_features, categorical_features)
    y_train = model_df.loc[train_mask, "log_hours_to_delivery"]
    pipeline.fit(X_train, y_train)
    return model_df, pipeline, feature_columns, numeric_features, categorical_features


def build_candidate_feature_frame(candidate_rows: pd.DataFrame, model_ready_order_line_df: pd.DataFrame) -> pd.DataFrame:
    static_cols = [
        "order_line_id",
        "quantity",
        "discount_rate",
        "promise_days",
        "plus",
        "user_level",
        "purchase_power",
        "city_level",
        "destination_region_id",
        "num_available_dcs_in_region",
        "package_count",
        "sku_type",
    ]
    static_lookup = model_ready_order_line_df[static_cols].drop_duplicates(subset=["order_line_id"]).copy()
    work = candidate_rows.drop(columns=["destination_region_id"], errors="ignore").merge(
        static_lookup,
        how="left",
        on="order_line_id",
        validate="m:1",
    )
    work["remote_fulfillment_flag"] = pd.to_numeric(work["candidate_remote_flag"], errors="coerce").fillna(0)
    work["inventory_at_fulfillment_dc"] = pd.to_numeric(work["inventory_available"], errors="coerce").fillna(0)
    work["promise_missing_flag"] = pd.to_numeric(work["promise_days"], errors="coerce").isna().astype(int)
    work["package_missing_flag"] = pd.to_numeric(work["package_count"], errors="coerce").isna().astype(int)
    return add_business_features(work)


def score_candidate_rows(
    candidate_rows: pd.DataFrame,
    model_ready_order_line_df: pd.DataFrame,
    fitted_pipeline,
    feature_columns: list[str],
    numeric_features: list[str],
    categorical_features: list[str],
) -> pd.DataFrame:
    work = build_candidate_feature_frame(candidate_rows, model_ready_order_line_df)
    model_input = prepare_model_matrix(work[feature_columns], numeric_features, categorical_features)
    pred_log = fitted_pipeline.predict(model_input)
    work["predicted_hours_to_delivery"] = np.clip(np.expm1(pred_log), a_min=0, a_max=None)
    return work


def build_sample_pool(order_line_df: pd.DataFrame) -> tuple[pd.DataFrame, list[pd.Timestamp], list[str]]:
    sample_dates = sorted(order_line_df["order_date"].dropna().unique())[:SAMPLE_DATES_N]
    date_filtered = order_line_df.loc[order_line_df["order_date"].isin(sample_dates)].copy()
    top_regions = (
        date_filtered["destination_region_id"]
        .astype("string")
        .value_counts(dropna=True)
        .head(TOP_REGIONS_N)
        .index
        .tolist()
    )
    sample_pool = (
        date_filtered.loc[date_filtered["destination_region_id"].astype("string").isin(top_regions)]
        .sort_values(["order_date", "order_line_id"])
        .drop_duplicates(subset=["order_line_id"])
        .head(MAX_SAMPLE_LINES)
        .copy()
    )
    return sample_pool, sample_dates, top_regions


def build_historical_status(candidate_pool: pd.DataFrame, sample_pool: pd.DataFrame) -> pd.DataFrame:
    work = candidate_pool.copy()
    work["historical_match_flag"] = work["candidate_dc"].astype("string").eq(work["dc_ori"].astype("string")).astype(int)

    historical_presence = (
        work.groupby("order_line_id", as_index=False)["historical_match_flag"]
        .max()
        .rename(columns={"historical_match_flag": "historical_in_candidate_set_flag"})
    )
    historical_feasible = (
        work.loc[work["inventory_available"].eq(1)]
        .groupby("order_line_id", as_index=False)["historical_match_flag"]
        .max()
        .rename(columns={"historical_match_flag": "historical_feasible_flag"})
    )
    feasible_choice = (
        work.loc[work["inventory_available"].eq(1)]
        .groupby("order_line_id", as_index=False)
        .agg(
            feasible_candidate_count=("candidate_dc", "nunique"),
            remote_option_values=("candidate_remote_flag", "nunique"),
        )
    )

    status = (
        sample_pool[["order_line_id"]]
        .drop_duplicates()
        .merge(historical_presence, on="order_line_id", how="left")
        .merge(historical_feasible, on="order_line_id", how="left")
        .merge(feasible_choice, on="order_line_id", how="left")
    )

    fill_cols = [
        "historical_in_candidate_set_flag",
        "historical_feasible_flag",
        "feasible_candidate_count",
        "remote_option_values",
    ]
    status[fill_cols] = status[fill_cols].fillna(0).astype(int)
    status["historical_missing_from_candidate_set_flag"] = 1 - status["historical_in_candidate_set_flag"]
    status["historical_present_but_inventory_blocked_flag"] = (
        status["historical_in_candidate_set_flag"].eq(1) & status["historical_feasible_flag"].eq(0)
    ).astype(int)
    status["multi_feasible_flag"] = status["feasible_candidate_count"].gt(1).astype(int)
    status["mixed_local_remote_flag"] = status["remote_option_values"].gt(1).astype(int)
    return status


def build_strict_baseline(analysis_input: pd.DataFrame, analysis_status: pd.DataFrame) -> pd.DataFrame:
    strict_ids = set(analysis_status.loc[analysis_status["historical_feasible_flag"].eq(1), "order_line_id"])
    baseline = (
        analysis_input.loc[
            analysis_input["order_line_id"].isin(strict_ids)
            & analysis_input["candidate_dc"].astype("string").eq(analysis_input["dc_ori"].astype("string")),
            ["order_line_id", "candidate_dc", "predicted_hours_to_delivery", "candidate_remote_flag"],
        ]
        .drop_duplicates(subset=["order_line_id"])
        .rename(
            columns={
                "candidate_dc": "baseline_candidate_dc",
                "predicted_hours_to_delivery": "baseline_predicted_hours",
                "candidate_remote_flag": "baseline_remote_flag",
            }
        )
    )
    return baseline


def build_order_groups(candidate_input: pd.DataFrame) -> dict[str, list[int]]:
    return {
        order_line_id: list(group)
        for order_line_id, group in candidate_input.groupby("order_line_id").groups.items()
    }


def solve_service_optimum(candidate_input: pd.DataFrame) -> tuple[pulp.LpProblem, pd.DataFrame, float]:
    work = candidate_input.copy().sort_values(["order_line_id", "candidate_dc"]).reset_index(drop=True)
    decision_index = list(work.index)
    order_groups = build_order_groups(work)

    x = pulp.LpVariable.dicts("assign_service", decision_index, 0, 1, cat="Binary")
    problem = pulp.LpProblem("service_optimum", pulp.LpMinimize)
    problem += pulp.lpSum(work.loc[i, "predicted_hours_to_delivery"] * x[i] for i in decision_index)

    for order_line_id, group_index in order_groups.items():
        problem += pulp.lpSum(x[i] for i in group_index) == 1, f"assign_once_{order_line_id}"

    problem.solve(pulp.PULP_CBC_CMD(msg=False, timeLimit=TIME_LIMIT_SECONDS))

    solution = work.copy()
    solution["selected"] = [int(round(pulp.value(x[i]))) for i in decision_index]
    solution = solution.loc[solution["selected"].eq(1)].copy()
    objective_value = float(pulp.value(problem.objective))
    return problem, solution.reset_index(drop=True), objective_value


def solve_lexicographic_remote_policy(
    candidate_input: pd.DataFrame,
    service_optimum_time: float,
    time_tolerance_total_hours: float,
) -> tuple[pulp.LpProblem, pd.DataFrame]:
    work = candidate_input.copy().sort_values(["order_line_id", "candidate_dc"]).reset_index(drop=True)
    decision_index = list(work.index)
    order_groups = build_order_groups(work)

    x = pulp.LpVariable.dicts("assign_lexi_remote", decision_index, 0, 1, cat="Binary")
    problem = pulp.LpProblem("lexicographic_remote_policy", pulp.LpMinimize)
    problem += pulp.lpSum(
        work.loc[i, "candidate_remote_flag"] * x[i]
        + EPSILON_TIE_BREAK * work.loc[i, "tie_break_rank"] * x[i]
        for i in decision_index
    )

    for order_line_id, group_index in order_groups.items():
        problem += pulp.lpSum(x[i] for i in group_index) == 1, f"assign_once_{order_line_id}"

    problem += (
        pulp.lpSum(work.loc[i, "predicted_hours_to_delivery"] * x[i] for i in decision_index)
        <= service_optimum_time + time_tolerance_total_hours,
        "time_tolerance_constraint",
    )

    problem.solve(pulp.PULP_CBC_CMD(msg=False, timeLimit=TIME_LIMIT_SECONDS))

    solution = work.copy()
    solution["selected"] = [int(round(pulp.value(x[i]))) for i in decision_index]
    solution = solution.loc[solution["selected"].eq(1)].copy()
    return problem, solution.reset_index(drop=True)


def build_workload_proxy_table(
    strict_baseline_df: pd.DataFrame,
    candidate_input: pd.DataFrame,
    gamma: float,
    load_floor: float,
) -> pd.DataFrame:
    candidate_dc_universe = (
        pd.Series(sorted(candidate_input["candidate_dc"].astype("string").dropna().unique()), name="candidate_dc")
        .to_frame()
    )
    baseline_load = (
        strict_baseline_df.groupby("baseline_candidate_dc", as_index=False)
        .size()
        .rename(columns={"baseline_candidate_dc": "candidate_dc", "size": "baseline_load"})
    )
    baseline_load["candidate_dc"] = baseline_load["candidate_dc"].astype("string")

    proxy = candidate_dc_universe.merge(baseline_load, on="candidate_dc", how="left")
    proxy["baseline_load"] = proxy["baseline_load"].fillna(0.0)
    proxy["proxy_cap"] = np.maximum(load_floor, (1 + gamma) * proxy["baseline_load"])
    return proxy


def solve_workload_proxy_policy(
    candidate_input: pd.DataFrame,
    proxy_df: pd.DataFrame,
    service_optimum_time: float,
    time_tolerance_total_hours: float,
) -> tuple[pulp.LpProblem, pulp.LpProblem, pd.DataFrame, float]:
    work = candidate_input.copy().sort_values(["order_line_id", "candidate_dc"]).reset_index(drop=True)
    decision_index = list(work.index)
    order_groups = build_order_groups(work)
    dc_list = proxy_df["candidate_dc"].astype("string").tolist()
    proxy_cap = proxy_df.set_index("candidate_dc")["proxy_cap"].to_dict()

    # Stage A: minimize total overload above proxy caps
    x_a = pulp.LpVariable.dicts("assign_workload_a", decision_index, 0, 1, cat="Binary")
    overload_a = pulp.LpVariable.dicts("overload_a", dc_list, lowBound=0, cat="Continuous")

    stage_a = pulp.LpProblem("workload_proxy_stage_a", pulp.LpMinimize)
    stage_a += pulp.lpSum(overload_a[dc] for dc in dc_list)

    for order_line_id, group_index in order_groups.items():
        stage_a += pulp.lpSum(x_a[i] for i in group_index) == 1, f"assign_once_a_{order_line_id}"

    stage_a += (
        pulp.lpSum(work.loc[i, "predicted_hours_to_delivery"] * x_a[i] for i in decision_index)
        <= service_optimum_time + time_tolerance_total_hours,
        "service_epsilon_constraint_a",
    )

    for dc in dc_list:
        dc_index = list(work.index[work["candidate_dc"].astype("string").eq(dc)])
        stage_a += (
            pulp.lpSum(x_a[i] for i in dc_index) <= float(proxy_cap[dc]) + overload_a[dc],
            f"proxy_cap_a_{dc}",
        )

    stage_a.solve(pulp.PULP_CBC_CMD(msg=False, timeLimit=TIME_LIMIT_SECONDS))
    best_total_overload = float(pulp.value(stage_a.objective))

    # Stage B: among overload-optimal solutions, minimize total predicted time
    x_b = pulp.LpVariable.dicts("assign_workload_b", decision_index, 0, 1, cat="Binary")
    overload_b = pulp.LpVariable.dicts("overload_b", dc_list, lowBound=0, cat="Continuous")

    stage_b = pulp.LpProblem("workload_proxy_stage_b", pulp.LpMinimize)
    stage_b += pulp.lpSum(
        work.loc[i, "predicted_hours_to_delivery"] * x_b[i]
        + EPSILON_TIE_BREAK * work.loc[i, "tie_break_rank"] * x_b[i]
        for i in decision_index
    )

    for order_line_id, group_index in order_groups.items():
        stage_b += pulp.lpSum(x_b[i] for i in group_index) == 1, f"assign_once_b_{order_line_id}"

    stage_b += (
        pulp.lpSum(work.loc[i, "predicted_hours_to_delivery"] * x_b[i] for i in decision_index)
        <= service_optimum_time + time_tolerance_total_hours,
        "service_epsilon_constraint_b",
    )
    stage_b += (
        pulp.lpSum(overload_b[dc] for dc in dc_list) <= best_total_overload + 1e-6,
        "overload_optimality_constraint",
    )

    for dc in dc_list:
        dc_index = list(work.index[work["candidate_dc"].astype("string").eq(dc)])
        stage_b += (
            pulp.lpSum(x_b[i] for i in dc_index) <= float(proxy_cap[dc]) + overload_b[dc],
            f"proxy_cap_b_{dc}",
        )

    stage_b.solve(pulp.PULP_CBC_CMD(msg=False, timeLimit=TIME_LIMIT_SECONDS))

    solution = work.copy()
    solution["selected"] = [int(round(pulp.value(x_b[i]))) for i in decision_index]
    solution = solution.loc[solution["selected"].eq(1)].copy()
    return stage_a, stage_b, solution.reset_index(drop=True), best_total_overload


def build_policy_comparison(
    strict_baseline_df: pd.DataFrame,
    policy_solution_df: pd.DataFrame,
    policy_name: str,
    policy_label: str,
    baseline_feasible_share: float,
) -> pd.DataFrame:
    compare = strict_baseline_df.merge(
        policy_solution_df[
            ["order_line_id", "candidate_dc", "predicted_hours_to_delivery", "candidate_remote_flag"]
        ].rename(
            columns={
                "candidate_dc": "chosen_candidate_dc",
                "predicted_hours_to_delivery": "optimized_predicted_hours",
                "candidate_remote_flag": "optimized_remote_flag",
            }
        ),
        on="order_line_id",
        how="inner",
        validate="1:1",
    )
    compare["policy_name"] = policy_name
    compare["policy_label"] = policy_label
    compare["hours_improvement"] = compare["baseline_predicted_hours"] - compare["optimized_predicted_hours"]
    compare["reassigned_flag"] = compare["baseline_candidate_dc"].astype("string").ne(compare["chosen_candidate_dc"].astype("string")).astype(int)
    compare["historical_baseline_feasible_share"] = baseline_feasible_share
    return compare


def summarize_policy(compare_df: pd.DataFrame) -> dict:
    reassigned_only = compare_df.loc[compare_df["reassigned_flag"].eq(1)].copy()
    return {
        "policy_name": compare_df["policy_name"].iloc[0],
        "policy_label": compare_df["policy_label"].iloc[0],
        "comparable_order_lines": len(compare_df),
        "historical_baseline_feasible_share": compare_df["historical_baseline_feasible_share"].iloc[0],
        "mean_predicted_fulfillment_hours": compare_df["optimized_predicted_hours"].mean(),
        "median_predicted_fulfillment_hours": compare_df["optimized_predicted_hours"].median(),
        "remote_fulfillment_share": compare_df["optimized_remote_flag"].mean(),
        "reassigned_order_lines": int(compare_df["reassigned_flag"].sum()),
        "reassignment_share": compare_df["reassigned_flag"].mean(),
        "average_improvement_vs_historical_baseline": compare_df["hours_improvement"].mean(),
        "average_improvement_reassigned_only": reassigned_only["hours_improvement"].mean() if len(reassigned_only) > 0 else np.nan,
    }


def build_workload_summary(
    policy_solution_df: pd.DataFrame,
    proxy_df: pd.DataFrame,
    policy_name: str,
    policy_label: str,
) -> tuple[pd.DataFrame, dict]:
    loads = (
        policy_solution_df.groupby("candidate_dc", as_index=False)
        .size()
        .rename(columns={"size": "policy_load"})
    )
    loads["candidate_dc"] = loads["candidate_dc"].astype("string")

    workload = proxy_df.copy()
    workload["candidate_dc"] = workload["candidate_dc"].astype("string")
    workload = workload.merge(loads, on="candidate_dc", how="left")
    workload["policy_load"] = workload["policy_load"].fillna(0.0)
    workload["load_gap_vs_baseline"] = workload["policy_load"] - workload["baseline_load"]
    workload["overload_above_proxy_cap"] = (workload["policy_load"] - workload["proxy_cap"]).clip(lower=0)
    workload["hitting_proxy_cap_flag"] = workload["policy_load"].ge(workload["proxy_cap"] - 1e-9).astype(int)
    workload["policy_name"] = policy_name
    workload["policy_label"] = policy_label

    summary = {
        "policy_name": policy_name,
        "policy_label": policy_label,
        "used_dcs": int(workload["policy_load"].gt(0).sum()),
        "max_dc_load": workload["policy_load"].max(),
        "mean_abs_load_gap_vs_baseline": workload["load_gap_vs_baseline"].abs().mean(),
        "total_overload_above_proxy_cap": workload["overload_above_proxy_cap"].sum(),
        "dcs_hitting_proxy_cap": int(workload["hitting_proxy_cap_flag"].sum()),
        "share_dcs_hitting_proxy_cap": workload["hitting_proxy_cap_flag"].mean(),
    }
    return workload, summary


def build_assignment_change_table(policy_solution_map: dict[str, pd.DataFrame]) -> pd.DataFrame:
    rows = []
    policy_names = list(policy_solution_map.keys())
    for i in range(len(policy_names)):
        for j in range(i + 1, len(policy_names)):
            policy_a = policy_names[i]
            policy_b = policy_names[j]
            map_a = policy_solution_map[policy_a].set_index("order_line_id")["candidate_dc"].astype("string")
            map_b = policy_solution_map[policy_b].set_index("order_line_id")["candidate_dc"].astype("string")
            aligned_a, aligned_b = map_a.align(map_b, join="inner")
            rows.append(
                {
                    "from_policy": policy_a,
                    "to_policy": policy_b,
                    "assignment_change_rate": aligned_a.ne(aligned_b).mean(),
                }
            )
    return pd.DataFrame(rows)


## 6. Sample definition and inputs

The optimization focuses on decision-relevant order lines rather than the full processed dataset.

Final sample logic:
- first 14 observed order dates,
- top 5 destination regions in those dates,
- up to 3,000 order lines,
- destination-region candidates only,
- feasible candidates only for optimization,
- and the main decision subset limited to order lines with more than one feasible candidate.

The baseline comparison then uses the strict historical-feasible subset inside that decision-relevant sample.


In [2]:
order_usecols = [
    "order_line_id",
    "order_date",
    "dc_des",
    "dc_ori",
    "hours_to_delivery",
    "delivered_by_jd_flag",
    "promise_days",
    "promise",
    "quantity",
    "discount_rate",
    "package_count",
    "inventory_at_dc_ori",
    "num_available_dcs_in_region",
    "plus",
    "user_level",
    "purchase_power",
    "city_level",
    "destination_region_id",
    "type",
]

candidate_usecols = [
    "order_line_id",
    "order_date",
    "dc_des",
    "dc_ori",
    "destination_region_id",
    "candidate_dc",
    "candidate_in_destination_region_flag",
    "candidate_remote_flag",
    "inventory_available",
]

order_line_df = pd.read_csv(ORDER_LINE_PATH, low_memory=False, usecols=lambda c: c in order_usecols)
candidate_df = pd.read_csv(CANDIDATE_PATH, low_memory=False, usecols=lambda c: c in candidate_usecols)

for frame in [order_line_df, candidate_df]:
    frame["order_line_id"] = frame["order_line_id"].astype("string")
    for col in ["dc_des", "dc_ori", "destination_region_id"]:
        frame[col] = normalize_code(frame[col])
    if "candidate_dc" in frame.columns:
        frame["candidate_dc"] = normalize_code(frame["candidate_dc"])
    frame["order_date"] = pd.to_datetime(frame["order_date"], errors="coerce").dt.normalize()

for col in ["candidate_in_destination_region_flag", "candidate_remote_flag", "inventory_available"]:
    candidate_df[col] = pd.to_numeric(candidate_df[col], errors="coerce").fillna(0).astype(int)

sample_pool, sample_dates, top_regions = build_sample_pool(order_line_df)
candidate_pool = candidate_df.loc[candidate_df["order_line_id"].isin(sample_pool["order_line_id"])].copy()
candidate_pool = candidate_pool.loc[candidate_pool["candidate_in_destination_region_flag"].eq(1)].copy()

model_ready_order_line_df, fitted_pipeline, feature_columns, numeric_features, categorical_features = fit_analysis3_proxy(order_line_df)
scored_candidate_pool = score_candidate_rows(
    candidate_rows=candidate_pool,
    model_ready_order_line_df=model_ready_order_line_df,
    fitted_pipeline=fitted_pipeline,
    feature_columns=feature_columns,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
)

scored_candidate_pool["candidate_remote_flag"] = pd.to_numeric(scored_candidate_pool["candidate_remote_flag"], errors="coerce").fillna(0).astype(int)
scored_candidate_pool["inventory_available"] = pd.to_numeric(scored_candidate_pool["inventory_available"], errors="coerce").fillna(0).astype(int)
scored_candidate_pool["predicted_hours_to_delivery"] = pd.to_numeric(scored_candidate_pool["predicted_hours_to_delivery"], errors="coerce")
scored_candidate_pool["package_count_proxy"] = pd.to_numeric(scored_candidate_pool["package_count"], errors="coerce")
scored_candidate_pool["quantity_proxy"] = pd.to_numeric(scored_candidate_pool["quantity"], errors="coerce")
scored_candidate_pool["package_complexity_proxy"] = (
    scored_candidate_pool["package_count_proxy"]
    .fillna(scored_candidate_pool["quantity_proxy"])
    .fillna(1)
    .clip(lower=1, upper=6)
)
scored_candidate_pool["extra_package_count"] = (scored_candidate_pool["package_complexity_proxy"] - 1).clip(lower=0)

historical_status_df = build_historical_status(candidate_pool=scored_candidate_pool, sample_pool=sample_pool)
feasible_candidate_pool = scored_candidate_pool.loc[
    scored_candidate_pool["inventory_available"].eq(1)
    & scored_candidate_pool["predicted_hours_to_delivery"].notna()
].copy()
feasible_candidate_pool["tie_break_rank"] = feasible_candidate_pool.groupby("order_line_id").cumcount() + 1

analysis_ids = historical_status_df.loc[historical_status_df["multi_feasible_flag"].eq(1), "order_line_id"]
analysis_input = (
    feasible_candidate_pool.loc[feasible_candidate_pool["order_line_id"].isin(analysis_ids)]
    .sort_values(["order_line_id", "candidate_dc"])
    .reset_index(drop=True)
)
analysis_status = historical_status_df.loc[historical_status_df["order_line_id"].isin(analysis_ids)].copy()

strict_baseline_df = build_strict_baseline(analysis_input=analysis_input, analysis_status=analysis_status)
strict_baseline_share = analysis_status["historical_feasible_flag"].mean()

sample_summary = pd.DataFrame(
    {
        "metric": [
            "sample_period_start",
            "sample_period_end",
            "top_destination_regions",
            "sample_pool_order_lines",
            "analysis_order_lines_multi_feasible_only",
            "strict_historical_comparable_order_lines",
            "mean_feasible_candidates_per_analysis_order_line",
        ],
        "value": [
            str(sample_dates[0])[:10],
            str(sample_dates[-1])[:10],
            ", ".join(top_regions),
            sample_pool["order_line_id"].nunique(),
            analysis_input["order_line_id"].nunique(),
            strict_baseline_df["order_line_id"].nunique(),
            analysis_status["feasible_candidate_count"].mean(),
        ],
    }
)

baseline_status_table = pd.DataFrame(
    {
        "status": [
            "historical_feasible",
            "historical_present_but_inventory_blocked",
            "historical_missing_from_candidate_set",
        ],
        "order_lines": [
            int(analysis_status["historical_feasible_flag"].sum()),
            int(analysis_status["historical_present_but_inventory_blocked_flag"].sum()),
            int(analysis_status["historical_missing_from_candidate_set_flag"].sum()),
        ],
    }
)
baseline_status_table["share"] = baseline_status_table["order_lines"] / baseline_status_table["order_lines"].sum()

display(sample_summary)
display(baseline_status_table)


,metric,value
0,sample_period_start,2018-03-01
1,sample_period_end,2018-03-14
2,top_destination_regions,"9, 2, 4, 5, 7"
3,sample_pool_order_lines,3000
4,analysis_order_lines_multi_feasible_only,1704
5,strict_historical_comparable_order_lines,1639
6,mean_feasible_candidates_per_analysis_order_line,4.4935


,status,order_lines,share
0,historical_feasible,1639,0.9619
1,historical_present_but_inventory_blocked,59,0.0346
2,historical_missing_from_candidate_set,6,0.0035


## 7. Solve formulation 1

Formulation 1 is the lexicographic service policy:

- Stage 1 minimizes total predicted fulfillment time.
- Stage 2 minimizes remote assignments while allowing only a very small aggregate service tolerance.

The tolerance is `50` predicted hours across the full analysis sample, which is roughly `0.03` hours per decision line on average.


In [3]:
service_stage1_problem, service_stage1_solution, service_optimum_time = solve_service_optimum(analysis_input)

lexicographic_problem, lexicographic_solution = solve_lexicographic_remote_policy(
    candidate_input=analysis_input,
    service_optimum_time=service_optimum_time,
    time_tolerance_total_hours=LEXI_TIME_TOLERANCE_TOTAL_HOURS,
)

service_stage1_compare = build_policy_comparison(
    strict_baseline_df=strict_baseline_df,
    policy_solution_df=service_stage1_solution,
    policy_name="service_stage1",
    policy_label="Pure service optimum",
    baseline_feasible_share=strict_baseline_share,
)
lexicographic_compare = build_policy_comparison(
    strict_baseline_df=strict_baseline_df,
    policy_solution_df=lexicographic_solution,
    policy_name="lexicographic_remote",
    policy_label="Lexicographic service then remote",
    baseline_feasible_share=strict_baseline_share,
)

formulation_1_stage_summary = pd.DataFrame(
    {
        "metric": [
            "stage_1_total_predicted_time",
            "stage_2_time_tolerance_total_hours",
            "stage_2_total_predicted_time",
            "stage_2_remote_share_all_analysis_lines",
            "assignment_change_stage1_to_stage2",
        ],
        "value": [
            service_optimum_time,
            LEXI_TIME_TOLERANCE_TOTAL_HOURS,
            lexicographic_solution["predicted_hours_to_delivery"].sum(),
            lexicographic_solution["candidate_remote_flag"].mean(),
            (
                service_stage1_solution.set_index("order_line_id")["candidate_dc"].astype("string")
                .align(lexicographic_solution.set_index("order_line_id")["candidate_dc"].astype("string"), join="inner")[0]
                .ne(
                    service_stage1_solution.set_index("order_line_id")["candidate_dc"].astype("string")
                    .align(lexicographic_solution.set_index("order_line_id")["candidate_dc"].astype("string"), join="inner")[1]
                )
                .mean()
            ),
        ],
    }
)

display(formulation_1_stage_summary)
display(pd.DataFrame([summarize_policy(lexicographic_compare)]))


,metric,value
0,stage_1_total_predicted_time,"41,013.6845"
1,stage_2_time_tolerance_total_hours,50.0000
2,stage_2_total_predicted_time,"41,051.7641"
3,stage_2_remote_share_all_analysis_lines,0.1620
4,assignment_change_stage1_to_stage2,0.1086


,policy_name,policy_label,comparable_order_lines,historical_baseline_feasible_share,mean_predicted_fulfillment_hours,median_predicted_fulfillment_hours,remote_fulfillment_share,reassigned_order_lines,reassignment_share,average_improvement_vs_historical_baseline,average_improvement_reassigned_only
0,lexicographic_remote,Lexicographic service then remote,1639,0.9619,23.8944,20.5172,0.1324,227,0.1385,0.3714,2.6814


## 8. Solve formulation 2

Formulation 2 is the workload-aware proxy policy:

- proxy cap per DC = `max(load_floor, (1 + gamma) * historical baseline load)`
- Stage A minimizes overload above those proxy caps
- subject to total predicted time staying within a small epsilon of the pure service optimum
- Stage B then minimizes total predicted time among overload-optimal solutions

This is not a true capacity model. It is a workload stress-test policy built from observed sample-period assignment volumes.


In [4]:
workload_proxy_df = build_workload_proxy_table(
    strict_baseline_df=strict_baseline_df,
    candidate_input=analysis_input,
    gamma=WORKLOAD_GAMMA,
    load_floor=WORKLOAD_FLOOR_LOAD,
)

workload_stage_a_problem, workload_stage_b_problem, workload_solution, best_total_overload = solve_workload_proxy_policy(
    candidate_input=analysis_input,
    proxy_df=workload_proxy_df,
    service_optimum_time=service_optimum_time,
    time_tolerance_total_hours=WORKLOAD_TIME_TOLERANCE_TOTAL_HOURS,
)

workload_compare = build_policy_comparison(
    strict_baseline_df=strict_baseline_df,
    policy_solution_df=workload_solution,
    policy_name="workload_proxy",
    policy_label="Workload-aware epsilon policy",
    baseline_feasible_share=strict_baseline_share,
)

formulation_2_summary = pd.DataFrame(
    {
        "metric": [
            "workload_gamma",
            "workload_floor_load",
            "service_tolerance_total_hours",
            "best_total_overload_above_proxy_caps",
            "workload_policy_total_predicted_time",
            "workload_policy_remote_share_all_analysis_lines",
        ],
        "value": [
            WORKLOAD_GAMMA,
            WORKLOAD_FLOOR_LOAD,
            WORKLOAD_TIME_TOLERANCE_TOTAL_HOURS,
            best_total_overload,
            workload_solution["predicted_hours_to_delivery"].sum(),
            workload_solution["candidate_remote_flag"].mean(),
        ],
    }
)

display(formulation_2_summary)
display(pd.DataFrame([summarize_policy(workload_compare)]))


,metric,value
0,workload_gamma,0.1000
1,workload_floor_load,10.0000
2,service_tolerance_total_hours,20.0000
3,best_total_overload_above_proxy_caps,157.6000
4,workload_policy_total_predicted_time,"41,033.3502"
5,workload_policy_remote_share_all_analysis_lines,0.1684


,policy_name,policy_label,comparable_order_lines,historical_baseline_feasible_share,mean_predicted_fulfillment_hours,median_predicted_fulfillment_hours,remote_fulfillment_share,reassigned_order_lines,reassignment_share,average_improvement_vs_historical_baseline,average_improvement_reassigned_only
0,workload_proxy,Workload-aware epsilon policy,1639,0.9619,23.8832,20.5172,0.1391,221,0.1348,0.3826,2.8375


## 9. Compare against historical baseline

The main comparison below reports the required outcome metrics against the strict historical baseline.

Policies shown:
- pure service optimum
- lexicographic service-then-remote
- workload-aware epsilon policy


In [5]:
policy_compare_map = {
    "service_stage1": service_stage1_compare,
    "lexicographic_remote": lexicographic_compare,
    "workload_proxy": workload_compare,
}

policy_summary_table = pd.DataFrame(
    [summarize_policy(policy_compare_map[key]) for key in policy_compare_map]
)

workload_distribution_by_policy = {}
workload_summary_rows = []
policy_solution_map = {
    "service_stage1": service_stage1_solution,
    "lexicographic_remote": lexicographic_solution,
    "workload_proxy": workload_solution,
}
policy_label_map = {
    "service_stage1": "Pure service optimum",
    "lexicographic_remote": "Lexicographic service then remote",
    "workload_proxy": "Workload-aware epsilon policy",
}

for policy_name, policy_solution_df in policy_solution_map.items():
    workload_detail, workload_summary = build_workload_summary(
        policy_solution_df=policy_solution_df,
        proxy_df=workload_proxy_df,
        policy_name=policy_name,
        policy_label=policy_label_map[policy_name],
    )
    workload_distribution_by_policy[policy_name] = workload_detail
    workload_summary_rows.append(workload_summary)

workload_summary_table = pd.DataFrame(workload_summary_rows)

display(policy_summary_table)
display(workload_summary_table)


,policy_name,policy_label,comparable_order_lines,historical_baseline_feasible_share,mean_predicted_fulfillment_hours,median_predicted_fulfillment_hours,remote_fulfillment_share,reassigned_order_lines,reassignment_share,average_improvement_vs_historical_baseline,average_improvement_reassigned_only
0,service_stage1,Pure service optimum,1639,0.9619,23.8712,20.5172,0.1330,363,0.2215,0.3946,1.7817
1,lexicographic_remote,Lexicographic service then remote,1639,0.9619,23.8944,20.5172,0.1324,227,0.1385,0.3714,2.6814
2,workload_proxy,Workload-aware epsilon policy,1639,0.9619,23.8832,20.5172,0.1391,221,0.1348,0.3826,2.8375


,policy_name,policy_label,used_dcs,max_dc_load,mean_abs_load_gap_vs_baseline,total_overload_above_proxy_cap,dcs_hitting_proxy_cap,share_dcs_hitting_proxy_cap
0,service_stage1,Pure service optimum,34,263.0000,22.3235,311.6000,19,0.5588
1,lexicographic_remote,Lexicographic service then remote,32,327.0000,12.5588,173.6000,18,0.5294
2,workload_proxy,Workload-aware epsilon policy,33,331.0000,11.9706,157.6000,18,0.5294


## 10. Compare policy trade-offs

This section asks whether different policy structures actually lead to different assignments and different operational implications.

The key questions are:
- do the policies choose different warehouses?
- how much service is given up, if any?
- how much workload pressure is relieved?


In [6]:
assignment_change_table = build_assignment_change_table(policy_solution_map)

dc_load_table = (
    workload_proxy_df[["candidate_dc", "baseline_load", "proxy_cap"]]
    .rename(columns={"baseline_load": "historical_baseline_load"})
    .copy()
)
for policy_name in policy_solution_map:
    dc_load_table = dc_load_table.merge(
        workload_distribution_by_policy[policy_name][["candidate_dc", "policy_load"]].rename(
            columns={"policy_load": f"{policy_name}_load"}
        ),
        on="candidate_dc",
        how="left",
    )
dc_load_table = dc_load_table.fillna(0.0)
top_dc_load_table = dc_load_table.sort_values("historical_baseline_load", ascending=False).head(12).copy()

display(assignment_change_table)
display(top_dc_load_table)

plot_df = policy_summary_table.copy()
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].bar(plot_df["policy_label"], plot_df["mean_predicted_fulfillment_hours"], color="#4c78a8")
axes[0].set_title("Mean predicted fulfillment hours")
axes[0].set_ylabel("Hours")
axes[0].tick_params(axis="x", rotation=20)

axes[1].bar(plot_df["policy_label"], plot_df["remote_fulfillment_share"], color="#e45756")
axes[1].set_title("Remote fulfillment share")
axes[1].set_ylabel("Share")
axes[1].tick_params(axis="x", rotation=20)

axes[2].bar(workload_summary_table["policy_label"], workload_summary_table["total_overload_above_proxy_cap"], color="#54a24b")
axes[2].set_title("Total overload above proxy cap")
axes[2].set_ylabel("Order-line load")
axes[2].tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()

top_plot = top_dc_load_table.copy()
x = np.arange(len(top_plot))
width = 0.25

plt.figure(figsize=(13, 5))
plt.bar(x - width, top_plot["historical_baseline_load"], width=width, label="Historical baseline", color="#9e9e9e")
plt.bar(x, top_plot["lexicographic_remote_load"], width=width, label="Lexicographic", color="#4c78a8")
plt.bar(x + width, top_plot["workload_proxy_load"], width=width, label="Workload-aware", color="#54a24b")
plt.xticks(x, top_plot["candidate_dc"], rotation=0)
plt.ylabel("Assigned order lines")
plt.title("Workload distribution for top baseline DCs")
plt.legend()
plt.tight_layout()
plt.show()


,from_policy,to_policy,assignment_change_rate
0,service_stage1,lexicographic_remote,0.1086
1,service_stage1,workload_proxy,0.1156
2,lexicographic_remote,workload_proxy,0.0106


,candidate_dc,historical_baseline_load,proxy_cap,service_stage1_load,lexicographic_remote_load,workload_proxy_load
33,9,402.0000,442.2000,263,327.0000,331.0000
21,5,284.0000,312.4000,253,269.0000,269.0000
15,4,279.0000,306.9000,239,263.0000,264.0000
5,2,246.0000,270.6000,170,211.0000,212.0000
31,7,137.0000,150.7000,76,97.0000,101.0000
9,27,58.0000,63.8000,104,93.0000,93.0000
6,20,37.0000,40.7000,49,43.0000,41.0000
17,43,22.0000,24.2000,39,29.0000,29.0000
10,28,21.0000,23.1000,36,33.0000,32.0000
11,31,20.0000,22.0000,33,29.0000,29.0000


C:\Users\Qiang Gao\AppData\Local\Temp\ipykernel_49156\3840063349.py:41: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
C:\Users\Qiang Gao\AppData\Local\Temp\ipykernel_49156\3840063349.py:56: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 11. Diagnose why gains are modest or meaningful

Stronger optimization does not guarantee large gains. This section checks which explanation is most supported:

- historical policy already strong,
- feasible candidate set still tight,
- workload proxy reduces movement,
- local-versus-remote differences already embedded in predicted time,
- or data structure still limits the size of possible gains.


In [7]:
within_order_dispersion = (
    analysis_input.groupby("order_line_id", as_index=False)
    .agg(
        feasible_candidate_count=("candidate_dc", "nunique"),
        min_predicted_hours=("predicted_hours_to_delivery", "min"),
        max_predicted_hours=("predicted_hours_to_delivery", "max"),
    )
)
within_order_dispersion["predicted_hour_spread"] = within_order_dispersion["max_predicted_hours"] - within_order_dispersion["min_predicted_hours"]

baseline_time_optimal_check = strict_baseline_df.merge(
    analysis_input.groupby("order_line_id", as_index=False)["predicted_hours_to_delivery"]
    .min()
    .rename(columns={"predicted_hours_to_delivery": "best_feasible_time"}),
    on="order_line_id",
    how="left",
    validate="1:1",
)
baseline_time_optimal_check["baseline_time_optimal_flag"] = np.isclose(
    baseline_time_optimal_check["baseline_predicted_hours"],
    baseline_time_optimal_check["best_feasible_time"],
    atol=1e-8,
)
baseline_time_optimal_check["time_gap_vs_best"] = (
    baseline_time_optimal_check["baseline_predicted_hours"] - baseline_time_optimal_check["best_feasible_time"]
)

best_local = (
    analysis_input.loc[analysis_input["candidate_remote_flag"].eq(0)]
    .groupby("order_line_id")["predicted_hours_to_delivery"]
    .min()
    .rename("best_local_hours")
)
best_remote = (
    analysis_input.loc[analysis_input["candidate_remote_flag"].eq(1)]
    .groupby("order_line_id")["predicted_hours_to_delivery"]
    .min()
    .rename("best_remote_hours")
)
local_remote_tradeoff = (
    analysis_status.loc[analysis_status["mixed_local_remote_flag"].eq(1), ["order_line_id"]]
    .merge(best_local, on="order_line_id", how="left")
    .merge(best_remote, on="order_line_id", how="left")
)
local_remote_tradeoff["remote_minus_local_hours"] = local_remote_tradeoff["best_remote_hours"] - local_remote_tradeoff["best_local_hours"]

diagnostic_table = pd.DataFrame(
    {
        "metric": [
            "analysis_order_lines",
            "strict_historical_comparable_order_lines",
            "share_baseline_already_time_optimal",
            "mean_time_gap_baseline_to_best",
            "mean_within_order_time_spread",
            "share_mixed_sets_where_best_local_beats_best_remote",
            "share_mixed_sets_where_remote_beats_local_by_more_than_0_1h",
            "service_policy_total_overload",
            "lexicographic_policy_total_overload",
            "workload_policy_total_overload",
        ],
        "value": [
            analysis_input["order_line_id"].nunique(),
            strict_baseline_df["order_line_id"].nunique(),
            baseline_time_optimal_check["baseline_time_optimal_flag"].mean(),
            baseline_time_optimal_check["time_gap_vs_best"].mean(),
            within_order_dispersion["predicted_hour_spread"].mean(),
            local_remote_tradeoff["remote_minus_local_hours"].ge(0).mean(),
            local_remote_tradeoff["remote_minus_local_hours"].lt(-0.1).mean(),
            workload_summary_table.loc[workload_summary_table["policy_name"].eq("service_stage1"), "total_overload_above_proxy_cap"].iloc[0],
            workload_summary_table.loc[workload_summary_table["policy_name"].eq("lexicographic_remote"), "total_overload_above_proxy_cap"].iloc[0],
            workload_summary_table.loc[workload_summary_table["policy_name"].eq("workload_proxy"), "total_overload_above_proxy_cap"].iloc[0],
        ],
    }
)

display(diagnostic_table)

plt.figure(figsize=(8, 4))
plt.hist(local_remote_tradeoff["remote_minus_local_hours"], bins=40, color="#72b7b2", edgecolor="white")
plt.title("Best remote minus best local predicted hours")
plt.xlabel("Remote - local hours")
plt.ylabel("Mixed local/remote order lines")
plt.tight_layout()
plt.show()

print("Diagnostic interpretation cues:")
if baseline_time_optimal_check["baseline_time_optimal_flag"].mean() >= 0.8:
    print("- Historical policy is already near the best feasible time option for most comparable order lines.")
if local_remote_tradeoff["remote_minus_local_hours"].ge(0).mean() >= 0.95:
    print("- Local candidates usually already beat remote candidates in predicted time, so remote-reduction policies have limited extra room.")
if workload_summary_table.loc[workload_summary_table["policy_name"].eq("workload_proxy"), "total_overload_above_proxy_cap"].iloc[0] < workload_summary_table.loc[workload_summary_table["policy_name"].eq("service_stage1"), "total_overload_above_proxy_cap"].iloc[0]:
    print("- The workload proxy does change the assignment pattern and relieves DC pressure, but only within a near-service frontier.")
if policy_summary_table["average_improvement_vs_historical_baseline"].max() < 1.0:
    print("- Average service gains remain modest, which suggests the candidate set and observed policy already leave limited optimization room.")


,metric,value
0,analysis_order_lines,"1,704.0000"
1,strict_historical_comparable_order_lines,"1,639.0000"
2,share_baseline_already_time_optimal,0.8658
3,mean_time_gap_baseline_to_best,0.3946
4,mean_within_order_time_spread,2.6096
5,share_mixed_sets_where_best_local_beats_best_r...,0.9993
6,share_mixed_sets_where_remote_beats_local_by_m...,0.0007
7,service_policy_total_overload,311.6000
8,lexicographic_policy_total_overload,173.6000
9,workload_policy_total_overload,157.6000


C:\Users\Qiang Gao\AppData\Local\Temp\ipykernel_49156\4057318634.py:84: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Diagnostic interpretation cues:
- Historical policy is already near the best feasible time option for most comparable order lines.
- Local candidates usually already beat remote candidates in predicted time, so remote-reduction policies have limited extra room.
- The workload proxy does change the assignment pattern and relieves DC pressure, but only within a near-service frontier.
- Average service gains remain modest, which suggests the candidate set and observed policy already leave limited optimization room.


## 12. Final interpretation for report and presentation

The stronger Part C now supports a more mature OR story than the previous simple assignment model:

- the baseline is genuinely historical,
- the lexicographic model formalizes service-first decision making,
- the workload-aware model formalizes a real service-versus-load trade-off,
- and different policy structures do produce different assignments.

At the same time, the notebook should still be interpreted honestly:

- if average gains remain modest, that does **not** mean the optimization failed,
- it can mean the historical policy is already strong,
- the candidate set is tight,
- and the Analysis 3 time proxy already captures much of the local-versus-remote effect.

For the final report, the key message is:

**A stronger OR formulation reveals policy trade-offs more clearly than the original simple model, but the data structure still limits how far reassignment can improve outcomes beyond the observed policy.**
